# Test Sub-Agent History Consistency & Statefulness

This notebook demonstrates how sub-agents (like the `RAGAgent`) persist their chat history across multiple calls from an orchestrator, and how the `_get_consistent_history` logic ensures valid message ordering even when history limits are applied.

In [1]:
import os
import sys
from typing import List, Dict, Any

# Ensure the project root is in the path
sys.path.append(os.path.abspath(".."))

from ai_tools.tools import LLMQuery
from agents.rag_agent import RAGAgent
from utils.config import settings
import logging

# Set up a logger to see the tool calls and history adjustments
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("TestNotebook")

## 1. Initialize RAGAgent and Orchestrator

We initialize the `RAGAgent` with a small `history_limit` (e.g., 5) to test the backtracking logic easily.

In [2]:
rag_agent = RAGAgent()
# Force a small history limit for testing backtracking logic
rag_agent.llm.history_limit = 5

logger.info(f"RAGAgent initialized with model: {rag_agent.model_name}")

# Create the orchestrator
orchestrator = LLMQuery(
    system_prompt="You are a Pokemon Master. Use the RAG agent for any lore or behavior questions.",
    tools=[rag_agent.as_tool()],
    model=settings.DEFAULT_MODEL
)

logger.info("Orchestrator initialized.")

[12:31:20] [RAGAgent] Agent 'RAGAgent' initialized with model 'openrouter/mistralai/mistral-small-2603'


INFO:RAGAgent:Agent 'RAGAgent' initialized with model 'openrouter/mistralai/mistral-small-2603'
INFO:TestNotebook:RAGAgent initialized with model: openrouter/mistralai/mistral-small-2603
INFO:TestNotebook:Orchestrator initialized.


## 2. First Call to RAGAgent

We ask the orchestrator about Charizard's behavior.

In [3]:
response1 = orchestrator.query("What is Charizard's behavior like?")
orchestrator.get_tool_responses()
print(f"Orchestrator Response 1: {orchestrator.response}")

print(f"\nRAGAgent History Length: {len(rag_agent.llm.chat_history)}")
for i, msg in enumerate(rag_agent.llm.chat_history):
    print(f"{i}: {msg['role']} - {str(msg.get('content'))[:50]}...")

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:31:38] [RAGAgent] QUERY: What is Charizard's behavior like?


INFO:RAGAgent:QUERY: What is Charizard's behavior like?
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:31:39] [RAGAgent] 🛠️  TOOL REQUESTED: query_database | Args: {"query": "Charizard behavior personality habits in battle and outside Pokemon games", "n_results": 3, "category": "pokemon", "filter_name": "Charizard"}


INFO:RAGAgent:🛠️  TOOL REQUESTED: query_database | Args: {"query": "Charizard behavior personality habits in battle and outside Pokemon games", "n_results": 3, "category": "pokemon", "filter_name": "Charizard"}


...

[12:31:40] [RAGAgent] TOOL OUTPUT (query_database): ### Entity: charizard
- **Category**: pokemon
- **ID**: 6
- **Generation**: Generation I

## Identity & Lore
Charizard (Pokedex #006) is the Flame Pokémon, an iconic Fire/Flying dual-type introduced in Generation I (Kanto). Residing primarily in rugged mountain habitats, its internal flame is hot enough to melt boulders, and it is known to unintentionally cause forest fires while searching for powerful opponents. Although not a Dragon-type by default, Charizard possesses significant latent poten... [truncated]


INFO:RAGAgent:TOOL OUTPUT (query_database): ### Entity: charizard
- **Category**: pokemon
- **ID**: 6
- **Generation**: Generation I

## Identity & Lore
Charizard (Pokedex #006) is the Flame Pokémon, an iconic Fire/Flying dual-type introduced in Generation I (Kanto). Residing primarily in rugged mountain habitats, its internal flame is hot enough to melt boulders, and it is known to unintentionally cause forest fires while searching for powerful opponents. Although not a Dragon-type by default, Charizard possesses significant latent poten... [truncated]
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:31:43] [RAGAgent] 🧠 RESPONSE: Charizard is a proud and strong-willed Pokémon known for its **dragon-like behavior**. It is highly **territorial** and **confident**, often seeking out powerful opponents to battle, especially in mountainous regions where it resides. Its **internal flame burns hotter when it is angered or excited**, and it has been known to **cause forest fires unintentionally** while searching for strong foes.

Despite its **intimidating presence**, Charizard is **loyal to trainers it trusts** and tends to be more **playful and affectionate** with them, often engaging in roughhousing or playful aerial maneuvers. However, it can be **stubborn and disobedient** if it feels mistreated or undervalued, especially if it perceives its trainer as weak.

In battle, it is **fierce and strategic**, preferring to **take down opponents quickly** with powerful **Fire and Flying-type moves** like **Flamethrower, Fire Blast, and Air Slash**. It is also known for its **enjoyment of s

INFO:RAGAgent:🧠 RESPONSE: Charizard is a proud and strong-willed Pokémon known for its **dragon-like behavior**. It is highly **territorial** and **confident**, often seeking out powerful opponents to battle, especially in mountainous regions where it resides. Its **internal flame burns hotter when it is angered or excited**, and it has been known to **cause forest fires unintentionally** while searching for strong foes.

Despite its **intimidating presence**, Charizard is **loyal to trainers it trusts** and tends to be more **playful and affectionate** with them, often engaging in roughhousing or playful aerial maneuvers. However, it can be **stubborn and disobedient** if it feels mistreated or undervalued, especially if it perceives its trainer as weak.

In battle, it is **fierce and strategic**, preferring to **take down opponents quickly** with powerful **Fire and Flying-type moves** like **Flamethrower, Fire Blast, and Air Slash**. It is also known for its **enjoyment of sunny wea

Orchestrator Response 1: Charizard is a proud, territorial, and strong-willed Pokémon with a dragon-like demeanor. It loves seeking powerful opponents—especially in mountainous regions—and its internal flame burns hotter when it's angry or excited, sometimes causing forest fires accidentally.  

Despite its intimidating presence, Charizard is loyal to trainers it trusts and can be playful and affectionate, enjoying roughhousing and aerial maneuvers. However, it can be stubborn or disobedient if it feels undervalued. In battle, it is fierce and strategic, favoring moves like Flamethrower and Air Slash, and it becomes more aggressive in sunny weather.

RAGAgent History Length: 4
0: user - What is Charizard's behavior like?...
1: assistant - None...
2: tool - ### Entity: charizard
- **Category**: pokemon
- **...
3: assistant - Charizard is a proud and strong-willed Pokémon kno...


## 3. Second Call to RAGAgent (Persistence Test)

We ask a follow-up about Charizard that implicitly relies on the previous context being known to the RAG agent (though the RAG agent is usually queried with full queries, this tests if it *can* remember).

In [4]:
response2 = orchestrator.query("Does it also mention anything about its flame?")
orchestrator.get_tool_responses()
print(f"Orchestrator Response 2: {orchestrator.response}")

print(f"\nRAGAgent History Length: {len(rag_agent.llm.chat_history)}")
for i, msg in enumerate(rag_agent.llm.chat_history):
    print(f"{i}: {msg['role']} - {str(msg.get('content'))[:50]}...")

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:31:53] [RAGAgent] QUERY: Charizard flame behavior and meaning


INFO:RAGAgent:QUERY: Charizard flame behavior and meaning
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:31:53] [RAGAgent] 🛠️  TOOL REQUESTED: query_database | Args: {"query": "Charizard flame behavior meaning color intensity heat in battle and outside Pokemon games", "n_results": 3, "category": "pokemon", "filter_name": "Charizard"}


INFO:RAGAgent:🛠️  TOOL REQUESTED: query_database | Args: {"query": "Charizard flame behavior meaning color intensity heat in battle and outside Pokemon games", "n_results": 3, "category": "pokemon", "filter_name": "Charizard"}


...

[12:31:54] [RAGAgent] TOOL OUTPUT (query_database): ### Entity: charizard
- **Category**: pokemon
- **ID**: 6
- **Generation**: Generation I

## Identity & Lore
Charizard (Pokedex #006) is the Flame Pokémon, an iconic Fire/Flying dual-type introduced in Generation I (Kanto). Residing primarily in rugged mountain habitats, its internal flame is hot enough to melt boulders, and it is known to unintentionally cause forest fires while searching for powerful opponents. Although not a Dragon-type by default, Charizard possesses significant latent poten... [truncated]


INFO:RAGAgent:TOOL OUTPUT (query_database): ### Entity: charizard
- **Category**: pokemon
- **ID**: 6
- **Generation**: Generation I

## Identity & Lore
Charizard (Pokedex #006) is the Flame Pokémon, an iconic Fire/Flying dual-type introduced in Generation I (Kanto). Residing primarily in rugged mountain habitats, its internal flame is hot enough to melt boulders, and it is known to unintentionally cause forest fires while searching for powerful opponents. Although not a Dragon-type by default, Charizard possesses significant latent poten... [truncated]
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:31:58] [RAGAgent] 🧠 RESPONSE: Charizard's **flame behavior** is directly tied to its **emotional state, health, and environment**, reflecting its **passion, power, and mood**:

1. **Flame Color and Heat**:
   - A **healthy Charizard** has a **blue flame** at the tip of its tail, indicating **high heat and strong control** over its fire.
   - When **angry, excited, or injured**, its flames turn **brighter and more intense**, often shifting to a **yellow or orange hue**, signifying **increased heat and power**.
   - A **weak or depressed Charizard** may have a **dull or flickering flame**, sometimes even **turning red**, which signals **low morale or poor health**.

2. **Flame Behavior in Battle**:
   - **Aggressive Charizard** unleashes **scorching, high-pressure flames** during battle, often melting boulders or causing **unintentional wildfires** in forests.
   - Under **harsh sunlight**, its flames **burn hotter and faster**, enhancing its Fire-type moves significantly.
   - When 

INFO:RAGAgent:🧠 RESPONSE: Charizard's **flame behavior** is directly tied to its **emotional state, health, and environment**, reflecting its **passion, power, and mood**:

1. **Flame Color and Heat**:
   - A **healthy Charizard** has a **blue flame** at the tip of its tail, indicating **high heat and strong control** over its fire.
   - When **angry, excited, or injured**, its flames turn **brighter and more intense**, often shifting to a **yellow or orange hue**, signifying **increased heat and power**.
   - A **weak or depressed Charizard** may have a **dull or flickering flame**, sometimes even **turning red**, which signals **low morale or poor health**.

2. **Flame Behavior in Battle**:
   - **Aggressive Charizard** unleashes **scorching, high-pressure flames** during battle, often melting boulders or causing **unintentional wildfires** in forests.
   - Under **harsh sunlight**, its flames **burn hotter and faster**, enhancing its Fire-type moves significantly.
   - When using **

Orchestrator Response 2: Yes! Charizard's flame is a key indicator of its physical and emotional state:

- **Flame color**: A healthy Charizard has a **blue flame** at the tip of its tail, signifying high heat and strong control. When angry or excited, the flame shifts to **brighter yellow or orange**, while a weak or depressed Charizard may have a **dull or flickering flame** (sometimes even red).

- **Flame behavior**: In battle, Charizard's flames intensify and can scorch or melt objects. When around trusted trainers, it may playfully curl its tail or emit small, controlled bursts as a sign of affection.

- **Symbolism**: The flame represents its inner strength, pride, and untamed spirit—losing control of it reflects losing control over itself.

So, Charizard's flame is more than just a weapon—it's a living barometer of its mood, health, and bond with its trainer.

RAGAgent History Length: 8
0: user - What is Charizard's behavior like?...
1: assistant - None...
2: tool - ### Entity:

## 4. Testing History Consistency (The Fix)

With `history_limit=5`, after a few turns of tool use, the RAG agent's history will be sliced. We can verify that it still starts with a `user` message due to our `_get_consistent_history` logic.

In [5]:
# Trigger enough turns to exceed history_limit=5
orchestrator.query("Tell me about Mewtwo's origin.")
orchestrator.get_tool_responses()

orchestrator.query("And what about Mew?")
orchestrator.get_tool_responses()

print(f"\nFinal RAGAgent History Length: {len(rag_agent.llm.chat_history)}")

# Now manually trigger a consistency check as if we were making a new query
consistent_hist = rag_agent.llm._get_consistent_history(5)
print(f"\nConsistent History (limit=5) starts with role: {consistent_hist[0]['role']}")
assert consistent_hist[0]['role'] == 'user', "History MUST start with a user message!"

for i, msg in enumerate(consistent_hist):
    print(f"{i}: {msg['role']} - {str(msg.get('content'))[:50]}...")

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:32:25] [RAGAgent] QUERY: Mewtwo origin and creation


INFO:RAGAgent:QUERY: Mewtwo origin and creation
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:32:26] [RAGAgent] 🛠️  TOOL REQUESTED: query_database | Args: {"query": "Mewtwo origin creation backstory who created it how it was made Pokemon games", "n_results": 3, "category": "pokemon", "filter_name": "Mewtwo"}


INFO:RAGAgent:🛠️  TOOL REQUESTED: query_database | Args: {"query": "Mewtwo origin creation backstory who created it how it was made Pokemon games", "n_results": 3, "category": "pokemon", "filter_name": "Mewtwo"}


...

[12:32:43] [RAGAgent] TOOL OUTPUT (query_database): ### Entity: mew
- **Category**: pokemon
- **ID**: 151
- **Generation**: Generation I

## Identity & Lore
Mew (Pokedex #151) is the Mythical **New Species Pokémon**, originally discovered in **Generation I (Kanto)**. Visually, it is a small, pink, bipedal feline with large blue eyes and a long, thin tail. Renowned as a "mirage" by experts due to its extreme rarity, Mew is said to possess the genetic composition of all Pokémon. This unique biology allows it to make itself invisible at will and, mo... [truncated]


INFO:RAGAgent:TOOL OUTPUT (query_database): ### Entity: mew
- **Category**: pokemon
- **ID**: 151
- **Generation**: Generation I

## Identity & Lore
Mew (Pokedex #151) is the Mythical **New Species Pokémon**, originally discovered in **Generation I (Kanto)**. Visually, it is a small, pink, bipedal feline with large blue eyes and a long, thin tail. Renowned as a "mirage" by experts due to its extreme rarity, Mew is said to possess the genetic composition of all Pokémon. This unique biology allows it to make itself invisible at will and, mo... [truncated]
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:32:46] [RAGAgent] 🧠 RESPONSE: Mew is a **Mythical Pokémon** originating from **Generation I (Kanto)**.

### **Origin and Lore**:
1. **Ancestor of All Pokémon**:
   - Mew is considered the **genetic ancestor** of all Pokémon, containing the **DNA of every species** in its makeup.
   - This is why it can **learn every move in the Pokémon games**, making it the most versatile Pokémon in existence.

2. **Discovery and Rarity**:
   - Mew was **discovered** in the **Guyana jungle** (as referenced in *Pokémon: The First Movie*).
   - It is so rare that it was long believed to be a **myth or urban legend**, earning it the title **"New Species Pokémon."**
   - Due to its elusive nature, it can **turn invisible at will**, reinforcing its mysterious reputation.

3. **Creation and Connection to Mewtwo**:
   - Scientists extracted Mew’s DNA to **create Mewtwo**, an artificial clone designed for **ultimate power**.
   - Unlike Mewtwo, Mew is **benevolent and gentle**, embodying the **original sp

INFO:RAGAgent:🧠 RESPONSE: Mew is a **Mythical Pokémon** originating from **Generation I (Kanto)**.

### **Origin and Lore**:
1. **Ancestor of All Pokémon**:
   - Mew is considered the **genetic ancestor** of all Pokémon, containing the **DNA of every species** in its makeup.
   - This is why it can **learn every move in the Pokémon games**, making it the most versatile Pokémon in existence.

2. **Discovery and Rarity**:
   - Mew was **discovered** in the **Guyana jungle** (as referenced in *Pokémon: The First Movie*).
   - It is so rare that it was long believed to be a **myth or urban legend**, earning it the title **"New Species Pokémon."**
   - Due to its elusive nature, it can **turn invisible at will**, reinforcing its mysterious reputation.

3. **Creation and Connection to Mewtwo**:
   - Scientists extracted Mew’s DNA to **create Mewtwo**, an artificial clone designed for **ultimate power**.
   - Unlike Mewtwo, Mew is **benevolent and gentle**, embodying the **original spirit of 


Final RAGAgent History Length: 16

Consistent History (limit=5) starts with role: user
0: user - Mewtwo origin and creation...
1: assistant - None...
2: tool - ### Entity: mewtwo
- **Category**: pokemon
- **ID*...
3: assistant - Mewtwo was **created artificially** through **gene...
4: user - Mew origin and lore...
5: assistant - None...
6: tool - ### Entity: mew
- **Category**: pokemon
- **ID**: ...
7: assistant - Mew is a **Mythical Pokémon** originating from **G...


In [8]:
orchestrator.query("What Pokemon did we talk about?")
orchestrator.get_tool_responses()

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


"We have discussed three Pokémon in this conversation:\n\n1. **Charizard** – A Fire/Flying-type Pokémon known for its dragon-like behavior, territorial nature, and its flame that reflects its emotional state.  \n2. **Mewtwo** – An artificial Legendary Pokémon created from Mew's DNA, with a tragic origin involving scientific experimentation.  \n3. **Mew** – The original Mythical Pokémon, considered the ancestor of all Pokémon, with psychic abilities and a benevolent nature.  \n\nLet me know if you'd like to explore any of them further!"

In [9]:
rag_agent.llm.query("What pokemon did we talk about?")

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


[12:37:48] [RAGAgent] 🛠️  TOOL REQUESTED: query_database | Args: {"query": "Mew Mewtwo Pokemon names", "n_results": 2, "category": "pokemon"}


INFO:RAGAgent:🛠️  TOOL REQUESTED: query_database | Args: {"query": "Mew Mewtwo Pokemon names", "n_results": 2, "category": "pokemon"}


''

In [10]:
rag_agent.llm.get_tool_responses()

[12:38:29] [RAGAgent] TOOL CALL (async): query_database | Args: {'query': 'Mew Mewtwo Pokemon names', 'n_results': 2, 'category': 'pokemon'}


INFO:RAGAgent:TOOL CALL (async): query_database | Args: {'query': 'Mew Mewtwo Pokemon names', 'n_results': 2, 'category': 'pokemon'}
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"


[12:38:30] [RAGAgent] TOOL OUTPUT (query_database): ### Entity: pawmi
- **Category**: pokemon
- **ID**: 921
- **Generation**: Generation IX

## Identity & Lore
Pawmi (Pokedex #921) is the Mouse Pokémon, a small quadruped creature introduced in Generation IX (Paldea). This yellow-furred Electric-type is characterized by its large, expressive ears and the thick fur on its neck that acts as an insulator. Biologically, Pawmi possesses underdeveloped electric sacs in its cheeks; it must rub its forepaws against its face to generate electricity through... [truncated]


INFO:RAGAgent:TOOL OUTPUT (query_database): ### Entity: pawmi
- **Category**: pokemon
- **ID**: 921
- **Generation**: Generation IX

## Identity & Lore
Pawmi (Pokedex #921) is the Mouse Pokémon, a small quadruped creature introduced in Generation IX (Paldea). This yellow-furred Electric-type is characterized by its large, expressive ears and the thick fur on its neck that acts as an insulator. Biologically, Pawmi possesses underdeveloped electric sacs in its cheeks; it must rub its forepaws against its face to generate electricity through... [truncated]
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


'Mew'

In [6]:
rag_agent.llm.chat_history

[{'role': 'user', 'content': "What is Charizard's behavior like?"},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'GA9H70D20',
    'function': {'arguments': '{"query": "Charizard behavior personality habits in battle and outside Pokemon games", "n_results": 3, "category": "pokemon", "filter_name": "Charizard"}',
     'name': 'query_database'},
    'type': 'function',
    'index': 0}]},
 {'role': 'tool',
  'content': "### Entity: charizard\n- **Category**: pokemon\n- **ID**: 6\n- **Generation**: Generation I\n\n## Identity & Lore\nCharizard (Pokedex #006) is the Flame Pokémon, an iconic Fire/Flying dual-type introduced in Generation I (Kanto). Residing primarily in rugged mountain habitats, its internal flame is hot enough to melt boulders, and it is known to unintentionally cause forest fires while searching for powerful opponents. Although not a Dragon-type by default, Charizard possesses significant latent potential, capable of undergoing two distinct Mega Evoluti